In [ ]:
import sys
from tqdm import tqdm

BASE_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr"
sys.path.append(BASE_PATH)
from src.data.kaldi_dataset import build_kaldi_datamodule

datamodule = build_kaldi_datamodule(
    "doreco",
    data_dir="/work/hdd/bbjs/shared/powsm/s2t1/dump/raw",
    dataset_config_path=f"{BASE_PATH}/configs/data/powsm_evalset_index.yaml",
    portable_wavscp=False,
    sampling_rate=16000,
    batch_size=2,
    num_workers=1,
)
datamodule.setup()

key2speech = {}
c = 0
for batch in tqdm(datamodule.predict_dataloader().dataset):
    key = batch["key"]
    speech = batch["speech"].cpu().numpy()
    key2speech[key] = speech
    c += 1
    if c % 100 == 0:
        break

In [ ]:
import json

prediction_path = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/inf_doreco_xeuspr/030000/transcription.json"

with open(prediction_path) as f:
    predictions = json.load(f)

pred = {"utt_id": [], "target": [], "prediction": []}
for idx in tqdm(predictions):
    item = predictions[idx]
    pred["utt_id"].append(item["passthrough"]["utt_id"])
    pred["target"].append(item["passthrough"]["target"])
    pred["prediction"].append(item["pred"][0]["processed_transcript"])

del predictions

print(f"loaded {len(pred['utt_id'])} predictions")

In [ ]:
import pandas as pd

PRED = {k: [] for k, _ in pred.items()}
PRED["speech"] = []
for i in tqdm(range(len(pred["utt_id"]))):
    utt_id = pred["utt_id"][i]
    if utt_id in key2speech:
        PRED["speech"].append(key2speech[utt_id])
        PRED["utt_id"].append(utt_id)
        PRED["target"].append(pred["target"][i])
        PRED["prediction"].append(pred["prediction"][i])

# del key2speech
df = pd.DataFrame(PRED)

In [ ]:
import numpy as np
import base64
import io
from scipy.io.wavfile import write
from IPython.display import HTML


# Helper to convert a single list of floats to an HTML player
def audio_tag(data):
    # Ensure it's a numpy array for processing
    audio_arr = np.array(data, dtype=np.float32)
    byte_io = io.BytesIO()
    # Replace 16000 with your actual sample rate
    write(byte_io, 16000, audio_arr)
    b64 = base64.b64encode(byte_io.getvalue()).decode("utf-8")
    return f'<audio controls style="width:120px; height:30px;"><source src="data:audio/wav;base64,{b64}" type="audio/wav"></audio>'


# Run on the first 10 rows
HTML(df.iloc[:10].to_html(escape=False, formatters={"speech": audio_tag}))

In [ ]:
import numpy as np
import panphon
import panphon.distance
from tqdm import tqdm
from tabulate import tabulate
import unicodedata
import string

# 1. Initialize Tools
ft = panphon.FeatureTable()
dst = panphon.distance.Distance()


def get_alignment_path(ref_segs, hyp_segs):
    """Standard Levenshtein DP to get the path."""
    R, H = len(ref_segs), len(hyp_segs)
    d = np.zeros((R + 1, H + 1))
    for i in range(R + 1):
        d[i, 0] = i
    for j in range(H + 1):
        d[0, j] = j

    for i in range(1, R + 1):
        for j in range(1, H + 1):
            cost = 0 if ref_segs[i - 1] == hyp_segs[j - 1] else 1
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + cost)

    path = []
    i, j = R, H
    while i > 0 or j > 0:
        if (
            i > 0
            and j > 0
            and d[i, j]
            == d[i - 1, j - 1] + (0 if ref_segs[i - 1] == hyp_segs[j - 1] else 1)
        ):
            path.append(("match/sub", i - 1, j - 1))
            i -= 1
            j -= 1
        elif i > 0 and d[i, j] == d[i - 1, j] + 1:
            path.append(("del", i - 1, None))
            i -= 1
        else:
            path.append(("ins", None, j - 1))
            j -= 1
    return path[::-1]


# Configuration
N_QUANTILES = 10
# We store totals to compute the final averages
totals = [{"fed_sum": 0.0, "ref_phones": 0} for _ in range(N_QUANTILES)]

# 2. Processing Loop
for i in tqdm(range(len(pred["utt_id"])), desc="Evaluating spans"):
    ref_txt = unicodedata.normalize(
        "NFD",
        pred["target"][i]
        .replace(" ", "")
        .translate(str.maketrans("", "", string.punctuation)),
    )
    hyp_txt = unicodedata.normalize(
        "NFD",
        pred["prediction"][i]
        .replace(" ", "")
        .translate(str.maketrans("", "", string.punctuation)),
    )

    ref_segs = ft.ipa_segs(ref_txt)
    hyp_segs = ft.ipa_segs(hyp_txt)
    n_ref = len(ref_segs)

    if n_ref < N_QUANTILES:
        continue  # Ensure enough segments to split

    path = get_alignment_path(ref_segs, hyp_segs)

    # Calculate quantile boundaries for Ground Truth
    # Indices for each quantile: [q_start, q_end)
    for q_idx in range(N_QUANTILES):
        q_start = (q_idx * n_ref) // N_QUANTILES
        q_end = ((q_idx + 1) * n_ref) // N_QUANTILES

        # 3. Find aligned span in predicted string
        # We look for all entries in the path where the reference index falls in [q_start, q_end)
        # We also include any 'ins' ops that occur immediately before or during this GT span.

        # Find path segment indices
        path_indices = []
        for idx, (op, r_i, h_i) in enumerate(path):
            # If the path entry involves a reference phone within our current quantile
            if r_i is not None and q_start <= r_i < q_end:
                path_indices.append(idx)

        if not path_indices:
            continue

        # Extract the sequence of predicted segments (hyp_segs) aligned to this path segment
        p_start_in_path = min(path_indices)
        p_end_in_path = max(path_indices)

        # Get all hyp indices within the range of the path corresponding to this GT quantile
        sub_hyp_indices = [
            path[k][2]
            for k in range(p_start_in_path, p_end_in_path + 1)
            if path[k][2] is not None
        ]

        # Prepare substrings
        gt_span_str = "".join(ref_segs[q_start:q_end])

        if sub_hyp_indices:
            h_min, h_max = min(sub_hyp_indices), max(sub_hyp_indices)
            hyp_span_str = "".join(hyp_segs[h_min : h_max + 1])
        else:
            hyp_span_str = ""

        # 4. Call FED distance error once for this full span
        # feature_edit_distance returns total distance for the string pair
        fed_val = dst.feature_edit_distance(gt_span_str, hyp_span_str)

        # 5. Store for PFER (fed / number of GT phones)
        totals[q_idx]["fed_sum"] += fed_val
        totals[q_idx]["ref_phones"] += q_end - q_start

# 4. Generate Report
table_data = []
for idx, t in enumerate(totals):
    n = t["ref_phones"]
    pfer = (t["fed_sum"] / n) if n > 0 else 0
    table_data.append(
        [
            f"Part {idx+1} ({(idx*100//N_QUANTILES)}%-{((idx+1)*100//N_QUANTILES)}%)",
            f"{pfer:.4f}",
            n,
        ]
    )

print("\n" + "=" * 60)
print("SPAN-BASED HYPOTHESIS TEST: FED DISTRIBUTION")
print("=" * 60)
print(tabulate(table_data, headers=["Portion", "Avg PFER (FED/N)", "Ref Phones"]))

In [ ]:
from collections import Counter
import pandas as pd

ft = panphon.FeatureTable()
phone_errors = Counter()
attr_errors = Counter()

for i in tqdm(range(len(pred["utt_id"])), desc="Analyzing Errors"):
    ref_segs = ft.ipa_segs(
        unicodedata.normalize("NFD", pred["target"][i].replace(" ", ""))
    )
    hyp_segs = ft.ipa_segs(
        unicodedata.normalize("NFD", pred["prediction"][i].replace(" ", ""))
    )

    path = get_alignment_path(ref_segs, hyp_segs)

    for op, r_idx, h_idx in path:
        if op == "match/sub" and ref_segs[r_idx] != hyp_segs[h_idx]:
            # Top mistaken phones
            phone_errors[f"{ref_segs[r_idx]} → {hyp_segs[h_idx]}"] += 1

            # Feature attribute errors
            r_vec = ft.word_to_vector_list(ref_segs[r_idx])[0]
            h_vec = ft.word_to_vector_list(hyp_segs[h_idx])[0]
            for val_r, val_h, name in zip(r_vec, h_vec, ft.names):
                if val_r != val_h:
                    attr_errors[name] += 1

        elif op == "del":
            phone_errors[f"{ref_segs[r_idx]} → [DEL]"] += 1
        elif op == "ins":
            phone_errors[f"[INS] → {hyp_segs[h_idx]}"] += 1

# Display Results
print("\n### TOP 10 PHONE MISTAKES")
print(pd.Series(dict(phone_errors.most_common(10))).to_string())

print("\n### TOP 10 MISTAKEN ATTRIBUTES (Feature Errors)")
print(pd.Series(dict(attr_errors.most_common(10))).to_string())

# Add this at the end of your analysis script
total_subs = sum(
    1 for op, r, h in path if op == "match/sub" and ref_segs[r] != hyp_segs[h]
)

print(f"\n### FEATURE ERROR DENSITY (Errors per Substitution)")
for attr, count in attr_errors.most_common(10):
    # This shows what % of mistakes involve this specific feature
    percentage = (count / total_subs) * 100 if total_subs > 0 else 0
    print(f"{attr:10} : {percentage:5.1f}% of substitutions")

# PR loss analysis

In [1]:
# NEEDED FOR RESAMPLING USING TORCHAUDIO
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# import torch

# torch.set_num_threads(1)
# torch.set_num_interop_threads(1)

In [2]:
import sys
from tqdm import tqdm

BASE_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr"

sys.path.append(BASE_PATH)

# ipapack
# from src.data.kaldi_pretraining_dataset import build_kaldi_datamodule
# datamodule = build_kaldi_datamodule(
#     "pr_fixed",
#     dataset_config_path=f"{BASE_PATH}/configs/data/ipapack_index.yaml",
#     vocab_file=f"{BASE_PATH}/src/model/xeusphoneme/resources/ipa_vocab.json",
#     batch_size=1,
#     num_workers=30,
#     limit_samples=None,
#     # filter_langs=["hin"],
#     read_asr_text=True,
# )

from src.data.kaldi_dataset import build_kaldi_datamodule

datamodule = build_kaldi_datamodule(
    "gmuaccent",
    data_dir="/work/hdd/bbjs/shared/powsm/s2t1/dump/raw",
    dataset_config_path=f"{BASE_PATH}/configs/data/powsm_evalset_index.yaml",
    portable_wavscp=False,
    sampling_rate=16000,
    batch_size=1,
    num_workers=1,
)

datamodule.setup()
dataloader = datamodule.test_dataloader()

print("Loaded dataset with length:", len(dataloader))

/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/.venv_dai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded dataset with length: 1242


In [3]:
import torch
from src.recipe.phone_recognition.model_module import PhoneRecognitionModel
from src.model.xeusphoneme.builders import build_xeus_pr_from_hf
from src.recipe.phone_recognition.greedy_ctc_strategy import GreedyCTCInference
import json

CKPT_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/train_ipapack_xeuspr/20251225_221757/checkpoints/step_030000.ckpt"
# CKPT_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/train_ipapack_xeuspr/20251231_100230/checkpoints/step_246268.ckpt"
VOCAB_FILE = f"{BASE_PATH}/src/model/xeusphoneme/resources/ipa_vocab.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device = ", device)

net = build_xeus_pr_from_hf(
    work_dir=f"{BASE_PATH}/exp/cache/xeus",
    checkpoint=CKPT_PATH,
    vocab_file=VOCAB_FILE,
    config_file=None,
    hf_repo="espnet/xeus",
)

model = PhoneRecognitionModel(net=net, optimizer=None)
model.set_inference_strategy(GreedyCTCInference)

model.to(device)
model.eval()

with open(VOCAB_FILE, "r") as f:
    vocab = json.load(f)
id2token = {k: v for v, k in vocab.items()}

Returning existing local_dir `/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/cache/xeus` as remote repo cannot be accessed in `snapshot_download` (None).


Using device =  cuda


In [4]:
from src.metrics.phone_recognition import PhoneRecognitionEvaluator

evaluator = PhoneRecognitionEvaluator()


def get_phone_str(token_ids):
    return "/".join([id2token[t] for t in token_ids if t in id2token])


N_SAMPLES = 150
n_data = len(dataloader)
print("Total data samples:", n_data, "sampling ", N_SAMPLES)
results = []
with torch.no_grad():
    for bidx, batch in tqdm(enumerate(dataloader), desc="Making predictions"):
        if bidx % (n_data // min(N_SAMPLES, n_data)) != 0:
            continue
        # assert batch size is 1
        batch = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }
        # print(batch)
        # out = model(batch)
        # batch_loss = out["loss"].item()
        batch_loss = 0  # IGNORE
        prediction = model.predict_step(batch, batch_idx=bidx)
        for i, key in enumerate(batch["keys"]):
            pred = prediction[i]["processed_transcript"]
            gt_str = batch["text"][i]
            if isinstance(gt_str, torch.Tensor):
                gt_str = gt_str.cpu().numpy()
                gt_phones = get_phone_str(gt_str)
            else:
                gt_phones = gt_str
            prmetrics, _ = evaluator.evaluate(
                {i: {"prediction": pred, "transcription": gt_phones.replace("/", "")}},
                compute_inventory=False,
            )
            asr_text = batch["asr_text"][i] if "asr_text" in batch else ""
            results.append(
                {
                    "key": key,
                    "loss": batch_loss / len(gt_str),
                    "speech": batch["speech"][i].cpu().numpy(),
                    "speech_length": batch["speech_length"][i].cpu().item(),
                    "wavpath": batch["wavpath"][i],
                    "phone_str": gt_phones,
                    "language": batch["lang_sym"][i],
                    "asr_text": asr_text,
                    "prediction": prediction[i]["predicted_transcript"],
                    "pr_metrics": prmetrics,
                }
            )
        # if bidx > 300:
        #     break
print(f"Collected {len(results)} samples for error analysis.")

Total data samples: 1242 sampling  150


Making predictions: 0it [00:00, ?it/s]WARNING:root:Flash Attention failed, falling back to default attention: FlashAttention only supports fp16, bf16, and fp8_e4m3 data type
Making predictions: 23it [00:08,  2.74it/s]

Making predictions: 109it [00:47,  2.50it/s]

Making predictions: 528it [03:34,  1.91it/s]

Making predictions: 1242it [08:28,  2.44it/s]

Collected 156 samples for error analysis.


In [5]:
import json
import os
import soundfile as sf
from pathlib import Path
from dataclasses import asdict
from tqdm import tqdm


def dump(
    results,
    filename,
    dump_dir="/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/cache/xeusresultsdump",
):
    dump_result = f"{dump_dir}/{filename}.jsonl"
    os.makedirs(os.path.dirname(dump_result), exist_ok=True)
    with open(dump_result, "w") as f:
        for r_ in tqdm(results):
            r = r_.copy()
            speech = r.pop("speech", None)
            if speech is not None:
                wpath = f"{dump_dir}/{r['key']}.wav"
                speech = speech[: r["speech_length"]]
                sf.write(wpath, speech, samplerate=16000)
                r["wavpath"] = Path(wpath).absolute().as_posix()
            r["pr_metrics"] = asdict(r["pr_metrics"])
            f.write(json.dumps(r, ensure_ascii=False, default=str))
            f.write("\n")


dump(results, filename="gmuaccent")

100%|██████████| 156/156 [00:01<00:00, 124.12it/s]


In [ ]:
results.sort(key=lambda x: x["pr_metrics"].FER, reverse=True)
threshold = 0
high_loss_samples = [r for r in results if r["loss"] > threshold]

print(f"Found {len(high_loss_samples)} samples with loss > {threshold}")


def display_audio(data):
    from IPython.display import Audio, display

    display(Audio(data, rate=16000))


for idx, sample in enumerate(high_loss_samples):
    print(f"Key: {sample['key']} | Language: {sample['language']}")
    print(f"Loss: {sample['loss']:.4f}")
    print(f"Phone GT: {sample['phone_str']}")
    print(f"Phone Pred: {sample['prediction']}")
    print(f"ASR GT: {sample['asr_text']}")
    print(f'PR Metrics [PFER]: {sample["pr_metrics"].FER}')
    print(f'PR Metrics [PER]: {sample["pr_metrics"].PER}')
    print(
        f'PR Metrics [SID]: {sample["pr_metrics"].SUB, sample["pr_metrics"].INS, sample["pr_metrics"].DEL}'
    )
    display_audio(sample["speech"])
    print("===" * 40)
    if idx >= 10:
        break

Found 22 samples with loss > 0
Key: aaaaa_cv_dev_0000000000000000008473130_pr | Language: hin
Loss: 1.1852
Phone GT: ʋ/ə/ɦ/b/i/ə/t/t͡ʃʰ/ä/ɦ/ɛ/u
Phone Pred: ʋ/e/b/i/a/t/t͡ʃʰ/ä/ɦ/ɛ
ASR GT: वह भी अच्छा है
PR Metrics [PFER]: 16.666666666666664
PR Metrics [PER]: 33.33333333333333
PR Metrics [SID]: (16.666666666666664, 0.0, 16.666666666666664)


Key: aaaaa_cv_dev_0000000000000000008472010_pr | Language: hin
Loss: 1.5568
Phone GT: ä/p/k/j/ä/k/ɛ/ɦ/n̪/e/k/i/k/o/ʃ/ɪ/ʃ/k/ə/ɾ/ɾ/ə/ɦ/e/ɦ/ɛ̃
Phone Pred: a/p/k/j/a/k/k/ɛ/h/n̪/e/k/i/k/o/ʃ/ɪ/ʃ/k/ʌ/r/ʌ/ẽ/ẽ
ASR GT: आप क्या कहने की कोशिश कर रहे हैं
PR Metrics [PFER]: 14.743589743589741
PR Metrics [PER]: 46.15384615384615
PR Metrics [SID]: (30.76923076923077, 3.8461538461538463, 11.538461538461538)


Key: aaaaa_cv_dev_0000000000000000008473370_pr | Language: hin
Loss: 2.6028
Phone GT: ʋ/ə/ɦ/ʋ/ə/ɦ/ä/ə/k/e/l̪/e/d͡ʒ/ä/n̪/e/l̪/ä/j/ə/q/b/ə/ɦ/ä/d̪/ʊ/ɾ/ɦ/ɛ/u
Phone Pred: ʋ/ɛ/h/ʋ/ʌ/h/ã/a/k/e/l/e/ɟ/a/n/e/l/a/ɪ/k/b/a/h/d/ʊ/r/h/ɛ
ASR GT: वह वहाँ अकेले जाने लायक बहादुर है
PR Metrics [PFER]: 11.895161290322578
PR Metrics [PER]: 70.96774193548387
PR Metrics [SID]: (61.29032258064516, 0.0, 9.67741935483871)


Key: aaaaa_cv_dev_0000000000000000008472570_pr | Language: hin
Loss: 7.0720
Phone GT: j/u/ɾ/e/n̪/i/j/ə̃/m/k/ɔ/ɾ/p/o/ɾ/e/ʃ/ə̃/n̪/ɔ/pʰ/ɪ̃/ɳ/ɖ/i/j/ä/l̪/ɪ/m/ɪ/ʈ/e/ɖ/m/ẽ/ʋ/ɛ/k/ẽ/n̪/s̪/i
Phone Pred: j/u/e/ɪ̃/n/i/ə/m/kʰ/ɔ/ɹ/p/ɜ˞/e/ɪ/ʃ/ə/n/ə/v/ɪ̃/n/d/i/ə/l/ɪ̃/m/ɪ/t/ɪ/d/m/e/v/e/ɪ/k/ə/n/s/i
ASR GT: यूरेनियम कॉरपोरेशन ऑफ इंडिया लिमिटेड में वैकेंसी
PR Metrics [PFER]: 11.724806201550386
PR Metrics [PER]: 69.76744186046511
PR Metrics [SID]: (62.7906976744186, 2.3255813953488373, 4.651162790697675)


Key: aaaaa_cv_dev_0000000000000000008472890_pr | Language: hin
Loss: 1.3997
Phone GT: ɛ/s̪/ä/m/ə/t̪/b/o/l̪/o/o
Phone Pred: ɛ/s/a/m/ə/t/b/o/l̪/o
ASR GT: ऐसा मत बोलो
PR Metrics [PFER]: 9.090909090909092
PR Metrics [PER]: 36.36363636363637
PR Metrics [SID]: (27.27272727272727, 0.0, 9.090909090909092)


...

Key: aaaaa_cv_dev_0000000000000000008473210_pr | Language: hin
Loss: 3.0490
Phone GT: ʈ/ɾ/ə̃/m/p/k/o/k/o/ɾ/ʈ/s̪/e/l̪/ə/ɡ/ä/t̪/ä/ɾ/d̪/u/s̪/ɾ/ä/d͡ʒ/ə/ʈ/k/ä/ʃ/ə/ɾ/ɳ/ä/ɾ/t̪/i/j/õ/k/i/ẽ/ɳ/ʈ/ɾ/i/p/ə/ɾ/ɾ/o/k/n̪/ə/ɦ/ĩ
Phone Pred: t/r/ʌ/m/p/k/o/k/o/ʈ/s/e/l/ʌ/ɡ/a/t/a/r/d/u/s/r/a/ɟ/ʌ/t/k/a/ʃ/ʌ/r/n/a/tʰ/j/õ/k/i/e/n/ʈ/r/i/p/ʌ/r/r/o/q/n/a/h/i/n
ASR GT: ट्रंप को कोर्ट से लगातार दूसरा झटका शरणार्थियों की एंट्री पर रोक नहीं
PR Metrics [PFER]: 8.479532163742691
PR Metrics [PER]: 66.66666666666666
PR Metrics [SID]: (59.64912280701754, 1.7543859649122806, 5.263157894736842)


Key: aaaaa_cv_dev_0000000000000000008472730_pr | Language: hin
Loss: 3.3965
Phone GT: ʋ/e/l̪/m/ẽ/ä/e/s̪/ä/n̪/s̪/d̪/õ/k/o/s̪/p/i/k/ə/ɾ/k/i/t͡ʃ/e/t̪/ä/ʋ/n̪/i/ʋ/ä/p/ə/s̪/d͡ʒ/ä/o/ʋ/ə/ɾ/n̪/ä/ɦ/o/ɡ/i/k/ä/ɾ/ɾ/ə/ʋ/ä/i
Phone Pred: ʋ/e/l/m/ẽ/a/e/s/a/n/s/ə/d/õ/k/o/ɪ/s/p/i/k/a/r/k/i/c/e/t/a/ʋ/n/i/ʋ/a/p/ə/s/ɟ/a/o/ʋ/a/r/n/a/h/o/ɡ/i/k/a/r/ʋ/a/i
ASR GT: वेल में आए सांसदों को स्पीकर की चेतावनी- वापस जाओ वर्ना होगी कार्रवाई
PR Metrics [PFER]: 8.409090909090908
PR Metrics [PER]: 54.54545454545454
PR Metrics [SID]: (47.27272727272727, 3.6363636363636362, 3.6363636363636362)


Key: aaaaa_cv_dev_0000000000000000008472250_pr | Language: hin
Loss: 1.0446
Phone GT: ʊ/b/ə/ɾ/k/ɛ/b/t͡ʃ/ä/l̪/ə/k/ʃ/ɪ/ʋ/ä/k/ʊ/m/ä/ɾ/k/e/kʰ/ɪ/l̪/ä/pʰ/ʃ/ɪ/k/ä/j/ə/t̪/d̪/ə/ɾ/d͡ʒ/k/ə/ɾ/ä/e/ɡ/i/ə/m/e/ɾ/ɪ/k/i/m/ə/ɦ/ɪ/l̪/ä
Phone Pred: o/b/ə/r/k/e/b/c/ä/l̪/ə/k/ʃ/ɪ/b/k/ʊ/m/ä/r/k/e/kʰ/ɪ/l̪/ä/b/ʃ/ə/k/ä/j/d/ə/r/d͡ʒ/k/ə/ɾ/ä/i/ɡ/i/ə/m/r/i/k/i/m/ə/ɦ/ɪ/l̪/ä
ASR GT: उबर कैब चालक शिव कुमार के खिलाफ शिकायत दर्ज कराएगी अमेरिकी महिला
PR Metrics [PFER]: 7.591807909604518
PR Metrics [PER]: 28.8135593220339
PR Metrics [SID]: (22.033898305084744, 0.0, 6.779661016949152)


Key: aaaaa_cv_dev_0000000000000000008472090_pr | Language: hin
Loss: 4.3139
Phone GT: ə/ɡ/ə/ɾ/ɦ/o/s̪/ə/k/t̪/ä/ɦ/ɛ/t̪/o/ɪ/s̪/ʃ/ə/n̪/ɪ/ʋ/ä/ɾ/j/ä/ɾ/ə/ʋ/ɪ/ʋ/ä/ɾ/o
Phone Pred: a/ɣ/a/r/h/o/s/ʌ/k/t/a/h/ɛ/t/o/ɪ/s/ʃ/ʌ/n/i/ʋ/a/r/j/a/r/ʌ/j/ʋ/a/r
ASR GT: अगर हो सकता है तो इस शनिवार या रविवार
PR Metrics [PFER]: 7.414215686274509
PR Metrics [PER]: 73.52941176470588
PR Metrics [SID]: (67.64705882352942, 0.0, 5.88235294117647)


Key: aaaaa_cv_dev_0000000000000000008473290_pr | Language: hin
Loss: 1.5642
Phone GT: t̪/ʊ̃/m/ə/p/n̪/e/s̪/t̪/ə̃/n̪/k/j/õ/t͡ʃʰ/ʊ/p/ä/t̪/i/ɦ/o
Phone Pred: t/u/m/m/a/p/n/e/s/t/a/n/k/j/õ/t͡ʃʰ/ə/p/a/t/i/h/o
ASR GT: तुम अपने स्तन क्यों छुपाती हो
PR Metrics [PFER]: 7.386363636363637
PR Metrics [PER]: 59.09090909090909
PR Metrics [SID]: (54.54545454545454, 4.545454545454546, 0.0)


In [ ]:
batch

In [ ]:
import numpy as np

pth = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/cache/lengths_pr.npy"
lengths = np.load(pth, allow_pickle=True)
lengths